In [1]:
import pandas as pd

In [2]:
df=pd.read_csv('anime.csv')

In [3]:
df.head()

,anime_id,name,genre,type,episodes,rating,members
0,32281,Kimi no Na wa.,"Drama, Romance, School, Supernatural",Movie,1,9.37,200630
1,5114,Fullmetal Alchemist: Brotherhood,"Action, Adventure, Drama, Fantasy, Magic, Mili...",TV,64,9.26,793665
2,28977,Gintama°,"Action, Comedy, Historical, Parody, Samurai, S...",TV,51,9.25,114262
3,9253,Steins;Gate,"Sci-Fi, Thriller",TV,24,9.17,673572
4,9969,Gintama&#039;,"Action, Comedy, Historical, Parody, Samurai, S...",TV,51,9.16,151266


In [4]:
df.isnull().sum()

anime_id      0
name          0
genre        62
type         25
episodes      0
rating      230
members       0
dtype: int64

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12294 entries, 0 to 12293
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   anime_id  12294 non-null  int64  
 1   name      12294 non-null  object 
 2   genre     12232 non-null  object 
 3   type      12269 non-null  object 
 4   episodes  12294 non-null  object 
 5   rating    12064 non-null  float64
 6   members   12294 non-null  int64  
dtypes: float64(1), int64(2), object(4)
memory usage: 672.5+ KB


In [6]:
df.shape

(12294, 7)

In [7]:
df['genre'] = df['genre'].fillna('Unknown')


In [8]:
df['type'] = df['type'].fillna(df['type'].mode()[0])


In [9]:
df['rating'] = df['rating'].fillna(df['rating'].mean())


In [10]:
df.isnull().sum()

anime_id    0
name        0
genre       0
type        0
episodes    0
rating      0
members     0
dtype: int64

In [11]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(stop_words='english')

genre_matrix = tfidf.fit_transform(df['genre'])


In [13]:
df['episodes'] = pd.to_numeric(df['episodes'], errors='coerce')


In [14]:
df['episodes'] = df['episodes'].fillna(df['episodes'].median())


In [15]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

numerical_features = df[['rating', 'episodes', 'members']]
numerical_scaled = scaler.fit_transform(numerical_features)


In [16]:
import numpy as np
from scipy.sparse import hstack

final_features = hstack([genre_matrix, numerical_scaled])


In [17]:
from sklearn.metrics.pairwise import cosine_similarity

cosine_sim = cosine_similarity(final_features, final_features)


In [18]:
def recommend_anime(title, top_n=5, threshold=0.3):
    if title not in df['name'].values:
        return "Anime not found"
    
    idx = df[df['name'] == title].index[0]
    similarity_scores = list(enumerate(cosine_sim[idx]))
    
    similarity_scores = sorted(similarity_scores, key=lambda x: x[1], reverse=True)
    
    recommendations = []
    for i, score in similarity_scores[1:]:
        if score >= threshold:
            recommendations.append(df.iloc[i]['name'])
        if len(recommendations) == top_n:
            break
    
    return recommendations


In [19]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)


In [20]:
def evaluate_recommendations(test_titles, threshold=0.3):
    y_true = []
    y_pred = []
    
    for title in test_titles:
        recs = recommend_anime(title, top_n=5, threshold=threshold)
        if isinstance(recs, list):
            y_true.append(1)
            y_pred.append(1 if len(recs) > 0 else 0)
    
    from sklearn.metrics import precision_score, recall_score, f1_score
    
    precision = precision_score(y_true, y_pred)
    recall = recall_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    
    return precision, recall, f1


In [21]:
precision, recall, f1 = evaluate_recommendations(test_df['name'].head(100))

print("Precision:", precision)
print("Recall:", recall)
print("F1-score:", f1)


Precision: 1.0
Recall: 1.0
F1-score: 1.0


#### Analysis and Area of Improvement

Precision → how many recommended anime were relevant

Recall → how many relevant anime were recommended

F1-score → balance between precision & recall

**Area of Improvement**
- Use user ratings matrix (collaborative filtering)
- Add anime synopsis text
- Use hybrid model (content + collaborative)
- Try Word2Vec / embeddings for genre similarity

### Interview Questions

#### Difference between User-based and Item-based Collaborative Filtering?

**User-Based Collaborative Filtering**

- Finds users with similar tastes

- Recommends items liked by similar users

- Problem: Not scalable for large users

**Item-Based Collaborative Filtering**

- Finds similar items

- Recommends items similar to what user liked

- More stable and scalable

#### What is Collaborative Filtering and How Does It Work?

**Collaborative Filtering is a recommendation technique that:**

Uses user behavior (ratings, likes, views)

Assumes users with similar behavior will like similar items

Works without item content

**Types:**

User-based

Item-based

Model-based (Matrix factorization)